# Roadmap Day 1 diagnostics

Builds fixed folds and measures original-coordinate size AP for the saved YOLO, RT-DETR-B, and RT-DETR-C checkpoints. It does not submit predictions.

In [ ]:
import importlib.util
import subprocess
import sys

packages = []
for name, spec in [('ultralytics', 'ultralytics==8.4.147'), ('pycocotools', 'pycocotools')]:
    if importlib.util.find_spec(name) is None:
        packages.append(spec)
if packages:
    subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)


In [ ]:
from pathlib import Path
Path('/kaggle/working/hsi_runner.py').write_text('"""Reproducible 16-band hyperspectral detector experiments and submission builder.\n\nSimple explanation:\n\nThis script:\n1. Reads the 16-band hyperspectral images.\n2. Reads the correct object boxes from XML files.\n3. Keeps some training images hidden for validation and holdout testing.\n4. Converts the data into the format YOLO expects.\n5. Trains one YOLO26m or RT-DETR-L object-detection model.\n6. Uses that model on the competition test images.\n7. Creates submission.csv only when holdout mAP50-95 is at least 0.50.\n\nImportant terms:\n- YOLO26m: an object detector. It predicts class, confidence, and box.\n- 16 bands: the 16 spectral channels in each competition image.\n- Validation: hidden training images used to check model progress.\n- Holdout: another hidden local set used as a safer final check.\n- Confidence: how sure the model is about one prediction.\n- Bounding box: rectangle around an object.\n\nThis version can read the competition raw PNG mosaics directly and applies the organizer-provided X2Cube conversion internally. It supports controlled aspect-ratio, cross-validation, spectral-augmentation, and same-checkpoint TTA experiments.\n"""\n\nfrom pathlib import Path\nimport argparse\nimport json\nimport random\nimport hashlib\nimport warnings\nimport importlib.metadata\nimport xml.etree.ElementTree as ET\n\nimport cv2\nfrom PIL import Image\nimport numpy as np\nimport pandas as pd\nimport yaml\nfrom sklearn.model_selection import StratifiedGroupKFold, train_test_split\n\n\n\ndef X2Cube(img, cellSize=4):\n    if img.ndim != 2 or any(n % cellSize for n in img.shape):\n        raise ValueError("Raw image sides must be divisible by 4")\n    B = [cellSize, cellSize]\n    skip = [cellSize, cellSize]\n    M, N = img.shape\n    col_extent = N - B[1] + 1\n    row_extent = M - B[0] + 1\n    start_idx = np.arange(B[0])[:, None] * N + np.arange(B[1])\n    didx = M * N * np.arange(1)\n    start_idx = (didx[:, None] + start_idx.ravel()).reshape((-1, B[0], B[1]))\n    offset_idx = np.arange(row_extent)[:, None] * N + np.arange(col_extent)\n    out = np.take(\n        img,\n        start_idx.ravel()[:, None] + offset_idx[::skip[0], ::skip[1]].ravel(),\n    )\n    out = np.transpose(out)\n    return out.reshape(M // cellSize, N // cellSize, cellSize * cellSize)\n\n\ndef read_classes(path):\n    names = [x.strip() for x in Path(path).read_text().splitlines() if x.strip()]\n    if len(names) != 18:\n        raise ValueError(f"Expected 18 classes, found {len(names)}")\n    return names\n\n\ndef read_cube(path):\n    path = Path(path)\n    if path.suffix.lower() == ".png":\n        raw = np.array(Image.open(path))\n        if raw.ndim != 2:\n            raise ValueError(f"{path} must be a 2D raw mosaic PNG, got {raw.shape}")\n        x = X2Cube(raw)\n    elif path.suffix.lower() == ".npy":\n        x = np.load(path)\n    elif path.suffix.lower() == ".npz":\n        z = np.load(path)\n        x = z[z.files[0]]\n    elif path.suffix.lower() in {".tif", ".tiff"}:\n        ok, frames = cv2.imreadmulti(str(path), flags=cv2.IMREAD_UNCHANGED)\n        if not ok:\n            raise ValueError(f"Cannot read {path}")\n        x = np.stack(frames, axis=-1)\n    else:\n        raise ValueError(f"Unsupported cube format: {path.suffix}")\n    x = np.asarray(x)\n    if not np.isfinite(x).all():\n        raise ValueError(f"Non-finite image values in {path}")\n    if x.ndim != 3:\n        raise ValueError(f"{path} must be 3D, got {x.shape}")\n    if x.shape[-1] == 16:\n        return x.astype(np.float32)\n    if x.shape[0] == 16:\n        return np.moveaxis(x, 0, -1).astype(np.float32)\n    raise ValueError(f"{path} does not contain 16 bands: {x.shape}")\n\n\ndef collect_cubes(root):\n    root = Path(root)\n    files = []\n    for ext in ("*.png", "*.npy", "*.npz", "*.tif", "*.tiff"):\n        files.extend(root.rglob(ext))\n    files = sorted({p.resolve() for p in files})\n    if not files:\n        raise FileNotFoundError(\n            "No 16-band cubes found. Run the organizer-compatible X2Cube conversion first."\n        )\n    return files\n\n\ndef band_limits(files, samples_per_image=256, seed=42):\n    rng = np.random.default_rng(seed)\n    chunks = []\n    for p in files:\n        x = read_cube(p).reshape(-1, 16)\n        n = min(samples_per_image, len(x))\n        idx = rng.choice(len(x), n, replace=False)\n        chunks.append(x[idx])\n    sample = np.concatenate(chunks, axis=0)\n    lo = np.percentile(sample, 0.5, axis=0)\n    hi = np.percentile(sample, 99.5, axis=0)\n    hi = np.maximum(hi, lo + 1e-6)\n    return lo, hi\n\n\ndef model_geometry(height, width, input_policy):\n    if input_policy == "native":\n        return {\n            "original_height": height,\n            "original_width": width,\n            "model_height": height,\n            "model_width": width,\n            "pad_top": 0,\n            "pad_left": 0,\n        }\n    if input_policy != "square_pad":\n        raise ValueError(f"Unknown input policy: {input_policy}")\n    side = max(height, width)\n    return {\n        "original_height": height,\n        "original_width": width,\n        "model_height": side,\n        "model_width": side,\n        "pad_top": (side - height) // 2,\n        "pad_left": (side - width) // 2,\n    }\n\n\ndef resolve_input_policy(family, input_policy):\n    if input_policy != "auto":\n        return input_policy\n    return "square_pad" if family == "rtdetr" else "native"\n\n\ndef prepare_cube(cube, lo, hi, input_policy, pipeline_state=None):\n    if pipeline_state is None:\n        x = np.clip((cube - lo) / (hi - lo), 0, 1)\n        x = np.rint(x * 255).astype(np.uint8)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import apply_pipeline\n        except ModuleNotFoundError:\n            from pipeline import apply_pipeline\n        x = apply_pipeline(cube, pipeline_state, output="uint8")\n    height, width = x.shape[:2]\n    geometry = model_geometry(height, width, input_policy)\n    if input_policy == "square_pad":\n        padded = np.zeros(\n            (geometry["model_height"], geometry["model_width"], 16),\n            dtype=np.uint8,\n        )\n        top = geometry["pad_top"]\n        left = geometry["pad_left"]\n        padded[top : top + height, left : left + width] = x\n        x = padded\n    return x, geometry\n\n\ndef write_tiff(cube, path, lo, hi, input_policy, pipeline_state=None):\n    x, geometry = prepare_cube(cube, lo, hi, input_policy, pipeline_state)\n    chw = np.moveaxis(x, -1, 0)\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    ok = cv2.imwritemulti(str(path), chw)\n    if not ok:\n        raise IOError(f"Failed to write {path}")\n    return geometry\n\n\ndef transform_rows(rows, geometry):\n    source_width = geometry["original_width"]\n    source_height = geometry["original_height"]\n    target_width = geometry["model_width"]\n    target_height = geometry["model_height"]\n    pad_left = geometry["pad_left"]\n    pad_top = geometry["pad_top"]\n    transformed = []\n    for cls, cx, cy, bw, bh in rows:\n        transformed.append(\n            (\n                cls,\n                (cx * source_width + pad_left) / target_width,\n                (cy * source_height + pad_top) / target_height,\n                bw * source_width / target_width,\n                bh * source_height / target_height,\n            )\n        )\n    return transformed\n\n\ndef restore_boxes(boxes, geometry):\n    boxes = np.asarray(boxes, dtype=np.float32).copy()\n    boxes[:, [0, 2]] -= geometry["pad_left"]\n    boxes[:, [1, 3]] -= geometry["pad_top"]\n    boxes[:, [0, 2]] = np.clip(boxes[:, [0, 2]], 0, geometry["original_width"])\n    boxes[:, [1, 3]] = np.clip(boxes[:, [1, 3]], 0, geometry["original_height"])\n    return boxes\n\n\ndef spectral_augment(cube, rng, gain_limit, tilt_limit, noise_std):\n    cube = cube.astype(np.float32, copy=True)\n    gain = rng.uniform(1.0 - gain_limit, 1.0 + gain_limit)\n    tilt = rng.uniform(-tilt_limit, tilt_limit)\n    wavelength_axis = np.linspace(-1.0, 1.0, 16, dtype=np.float32)\n    cube *= gain * (1.0 + tilt * wavelength_axis[None, None, :])\n    if noise_std > 0:\n        scale = np.maximum(cube.std(axis=(0, 1)), 1e-6)\n        noise = rng.normal(0.0, noise_std, size=cube.shape).astype(np.float32)\n        cube += noise * scale[None, None, :]\n    return cube\n\n\ndef stable_rng(seed, stem, copy_index):\n    digest = hashlib.sha256(f"{seed}:{stem}:{copy_index}".encode()).digest()\n    return np.random.default_rng(int.from_bytes(digest[:8], "little"))\n\n\ndef expand_rgb_stem(target_model, source_model):\n    import torch\n\n    target_layers = [layer for layer in target_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    source_layers = [layer for layer in source_model.modules() if isinstance(layer, torch.nn.Conv2d)]\n    target = next((layer for layer in target_layers if layer.in_channels == 16), None)\n    source = next(\n        (\n            layer\n            for layer in source_layers\n            if layer.in_channels == 3\n            and target is not None\n            and layer.out_channels == target.out_channels\n            and layer.kernel_size == target.kernel_size\n        ),\n        None,\n    )\n    if target is None or source is None:\n        raise RuntimeError("Could not identify compatible 3-channel and 16-channel input stems")\n    expanded = source.weight.detach().mean(dim=1, keepdim=True).repeat(1, 16, 1, 1)\n    expanded *= 3.0 / 16.0\n    target.weight.data.copy_(expanded.to(device=target.weight.device, dtype=target.weight.dtype))\n    if target.bias is not None and source.bias is not None:\n        target.bias.data.copy_(source.bias.detach().to(device=target.bias.device, dtype=target.bias.dtype))\n    print("Initialized the 16-channel input stem from RGB weights with activation-scale correction")\n\n\ndef trainer_class(args):\n    if args.stem_init == "random":\n        return None\n    if args.family == "rtdetr":\n        from ultralytics.models.rtdetr.train import RTDETRTrainer\n\n        class HSITrainer(RTDETRTrainer):\n            def get_model(self, cfg=None, weights=None, verbose=True):\n                model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n                if weights is None:\n                    raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n                expand_rgb_stem(model, weights)\n                return model\n\n        return HSITrainer\n    from ultralytics.models.yolo.detect import DetectionTrainer\n\n    class HSITrainer(DetectionTrainer):\n        def get_model(self, cfg=None, weights=None, verbose=True):\n            model = super().get_model(cfg=cfg, weights=weights, verbose=verbose)\n            if weights is None:\n                raise RuntimeError("rgb_mean stem initialization requires pretrained weights")\n            expand_rgb_stem(model, weights)\n            return model\n\n    return HSITrainer\n\n\ndef parse_voc(xml_path, class_to_id):\n    root = ET.parse(xml_path).getroot()\n    size = root.find("size")\n    width = int(float(size.findtext("width")))\n    height = int(float(size.findtext("height")))\n    rows = []\n    for obj in root.findall("object"):\n        name = obj.findtext("name").strip()\n        if name not in class_to_id:\n            raise ValueError(f"Unknown class {name} in {xml_path}")\n        b = obj.find("bndbox")\n        x1 = float(b.findtext("xmin"))\n        y1 = float(b.findtext("ymin"))\n        x2 = float(b.findtext("xmax"))\n        y2 = float(b.findtext("ymax"))\n        x1 = min(max(x1, 0.0), width)\n        x2 = min(max(x2, 0.0), width)\n        y1 = min(max(y1, 0.0), height)\n        y2 = min(max(y2, 0.0), height)\n        if x2 <= x1 or y2 <= y1:\n            continue\n        cx = ((x1 + x2) / 2) / width\n        cy = ((y1 + y2) / 2) / height\n        bw = (x2 - x1) / width\n        bh = (y2 - y1) / height\n        rows.append((class_to_id[name], cx, cy, bw, bh))\n    return width, height, rows\n\n\ndef annotation_map(xml_root):\n    xmls = list(Path(xml_root).rglob("*.xml"))\n    if not xmls:\n        raise FileNotFoundError(f"No VOC XML files found under {xml_root}")\n    return unique_stems(xmls)\n\n\ndef rarest_image_label(stems, boxes_by_stem):\n    counts = {}\n    for stem in stems:\n        for row in boxes_by_stem[stem]:\n            cls = int(row[0])\n            counts[cls] = counts.get(cls, 0) + 1\n    labels = []\n    for stem in stems:\n        classes = {int(row[0]) for row in boxes_by_stem[stem]}\n        if not classes:\n            labels.append(-1)\n        else:\n            labels.append(min(classes, key=lambda c: counts[c]))\n    return labels\n\n\ndef safe_split(stems, boxes_by_stem, seed):\n    labels = rarest_image_label(stems, boxes_by_stem)\n    try:\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            stratify=labels,\n        )\n        temp_labels = rarest_image_label(temp, boxes_by_stem)\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            stratify=temp_labels,\n        )\n    except ValueError as exc:\n        warnings.warn(f"Class-balanced split failed; using random split: {exc}")\n        train, temp = train_test_split(\n            stems,\n            test_size=0.20,\n            random_state=seed,\n            shuffle=True,\n        )\n        val, holdout = train_test_split(\n            temp,\n            test_size=0.50,\n            random_state=seed,\n            shuffle=True,\n        )\n    return sorted(train), sorted(val), sorted(holdout)\n\n\ndef read_groups(path, stems):\n    if not path:\n        return {stem: stem for stem in stems}, "image_id_only"\n    frame = pd.read_csv(path, dtype=str)\n    expected = {"image_id", "group_id"}\n    if set(frame.columns) != expected:\n        raise ValueError(f"Groups file columns must be {sorted(expected)}")\n    if frame["image_id"].duplicated().any():\n        raise ValueError("Groups file contains duplicate image_id values")\n    groups = dict(zip(frame["image_id"], frame["group_id"]))\n    missing = sorted(set(stems) - set(groups))\n    if missing:\n        raise ValueError(f"Groups file is missing {len(missing)} training IDs")\n    return {stem: groups[stem] for stem in stems}, str(Path(path))\n\n\ndef cv_split(stems, boxes_by_stem, groups_by_stem, seed, folds, fold_index):\n    if folds < 2:\n        raise ValueError("cv_folds must be at least 2")\n    if not 0 <= fold_index < folds:\n        raise ValueError(f"cv_fold_index must be in 0..{folds - 1}")\n    labels = np.asarray(rarest_image_label(stems, boxes_by_stem))\n    groups = np.asarray([groups_by_stem[stem] for stem in stems])\n    holdout_splitter = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=seed)\n    design_idx, holdout_idx = next(holdout_splitter.split(stems, labels, groups))\n    design_stems = np.asarray(stems)[design_idx]\n    design_labels = labels[design_idx]\n    design_groups = groups[design_idx]\n    fold_splitter = StratifiedGroupKFold(n_splits=folds, shuffle=True, random_state=seed + 1)\n    selected = list(fold_splitter.split(design_stems, design_labels, design_groups))[fold_index]\n    train_idx, val_idx = selected\n    return (\n        sorted(design_stems[train_idx].tolist()),\n        sorted(design_stems[val_idx].tolist()),\n        sorted(np.asarray(stems)[holdout_idx].tolist()),\n    )\n\n\ndef load_split_manifest(path, stems):\n    split_map = json.loads(Path(path).read_text())\n    if set(split_map) != {"train", "val", "test"}:\n        raise ValueError("Split manifest must contain train, val, and test")\n    flattened = [str(stem) for split in split_map.values() for stem in split]\n    if len(flattened) != len(set(flattened)):\n        raise ValueError("Split manifest contains duplicate IDs across splits")\n    if set(flattened) != set(stems):\n        raise ValueError("Split manifest IDs do not exactly match usable training IDs")\n    return {key: sorted(map(str, value)) for key, value in split_map.items()}\n\n\ndef assert_group_isolation(split_map, groups_by_stem):\n    group_splits = {}\n    for split, stems in split_map.items():\n        for stem in stems:\n            group = groups_by_stem[stem]\n            if group in group_splits and group_splits[group] != split:\n                raise ValueError(f"Group {group} leaks across {group_splits[group]} and {split}")\n            group_splits[group] = split\n\n\ndef write_label(path, rows):\n    Path(path).parent.mkdir(parents=True, exist_ok=True)\n    text = "\\n".join(\n        f"{int(c)} {cx:.8f} {cy:.8f} {w:.8f} {h:.8f}"\n        for c, cx, cy, w, h in rows\n    )\n    Path(path).write_text(text + ("\\n" if text else ""))\n\n\ndef image_id_value(stem):\n    return int(stem) if stem.isdigit() else stem\n\n\ndef parse_class_ids(value):\n    if not value:\n        return []\n    ids = sorted({int(item) for item in value.split(",")})\n    if any(class_id < 0 or class_id > 17 for class_id in ids):\n        raise argparse.ArgumentTypeError("Class IDs must be between 0 and 17")\n    return ids\n\n\ndef build_dataset(args):\n    work = Path(args.workdir)\n    cache_root = Path(args.cache_dir) if args.cache_dir else work\n    prepared = cache_root / "dataset"\n    if prepared.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {prepared}")\n    names = read_classes(args.class_file)\n    class_to_id = {name: i for i, name in enumerate(names)}\n    train_cubes = collect_cubes(args.train_cubes)\n    test_cubes = collect_cubes(args.test_cubes)\n    ranking_cubes = collect_cubes(args.ranking_cubes) if args.ranking_cubes else []\n    xmls = annotation_map(args.train_xml)\n    cube_by_stem = unique_stems(train_cubes)\n    inference_by_stem = unique_stems(test_cubes + ranking_cubes)\n    if set(cube_by_stem) != set(xmls):\n        raise ValueError("Training image and XML IDs do not match")\n    stems = sorted(set(cube_by_stem) & set(xmls))\n    bad_stems = {"1227", "1836", "1855"}\n    stems = [s for s in stems if s not in bad_stems]\n    print(f"Skipping mismatched images: {sorted(bad_stems)}")\n\n    if not stems:\n        raise RuntimeError("No matching training cube/XML stems found")\n\n    boxes_by_stem = {}\n    dims_by_stem = {}\n    for stem in stems:\n        w, h, rows = parse_voc(xmls[stem], class_to_id)\n        boxes_by_stem[stem] = rows\n        dims_by_stem[stem] = (w, h)\n\n    groups_by_stem, groups_source = read_groups(args.groups_file, stems)\n    if args.split_manifest:\n        split_map = load_split_manifest(args.split_manifest, stems)\n    elif args.cv_fold_index is not None:\n        train_stems, val_stems, holdout_stems = cv_split(\n            stems,\n            boxes_by_stem,\n            groups_by_stem,\n            args.seed,\n            args.cv_folds,\n            args.cv_fold_index,\n        )\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    else:\n        train_stems, val_stems, holdout_stems = safe_split(stems, boxes_by_stem, args.seed)\n        split_map = {"train": train_stems, "val": val_stems, "test": holdout_stems}\n    assert_group_isolation(split_map, groups_by_stem)\n    train_stems = split_map["train"]\n    val_stems = split_map["val"]\n    holdout_stems = split_map["test"]\n    normalization = getattr(args, "normalization", "N0")\n    spectral_preprocess = getattr(args, "spectral_preprocess", "SP0")\n    pipeline_state = None\n    if normalization == "N0" and spectral_preprocess == "SP0":\n        lo, hi = band_limits([cube_by_stem[s] for s in train_stems], seed=args.seed)\n    else:\n        try:\n            from Experiments.preprocessing.pipeline import fit_pipeline, save_pipeline_state\n        except ModuleNotFoundError:\n            from pipeline import fit_pipeline, save_pipeline_state\n        pipeline_state = fit_pipeline(\n            (read_cube(cube_by_stem[s]) for s in train_stems),\n            normalization,\n            spectral_method=spectral_preprocess,\n            seed=args.seed,\n            samples_per_cube=getattr(args, "normalization_samples_per_image", 256),\n            mnf_components=getattr(args, "mnf_components", None),\n            savgol_window=getattr(args, "savgol_window", 5),\n            savgol_order=getattr(args, "savgol_order", 2),\n        )\n        lo = hi = None\n        save_pipeline_state(pipeline_state, work / "spectral_pipeline_state.json")\n    input_policy = resolve_input_policy(args.family, args.input_policy)\n    work.mkdir(parents=True, exist_ok=True)\n    (work / "split_manifest.json").write_text(json.dumps(split_map, indent=2))\n    train_images_written = 0\n    for split, split_stems in split_map.items():\n        for stem in split_stems:\n            cube = read_cube(cube_by_stem[stem])\n            h, w = cube.shape[:2]\n            ann_w, ann_h = dims_by_stem[stem]\n            if (w, h) != (ann_w, ann_h):\n                raise ValueError(\n                    f"Spatial mismatch for {stem}: cube {(w, h)} vs annotation {(ann_w, ann_h)}"\n                )\n            geometry = write_tiff(\n                cube,\n                prepared / "images" / split / f"{stem}.tiff",\n                lo,\n                hi,\n                input_policy,\n                pipeline_state,\n            )\n            rows = transform_rows(boxes_by_stem[stem], geometry)\n            write_label(prepared / "labels" / split / f"{stem}.txt", rows)\n            if split != "train":\n                continue\n            train_images_written += 1\n            classes = {int(row[0]) for row in boxes_by_stem[stem]}\n            selected = not args.spectral_aug_classes or classes.intersection(args.spectral_aug_classes)\n            if not selected:\n                continue\n            for copy_index in range(args.spectral_aug_copies):\n                suffix = f"{stem}_hsaug{copy_index + 1}"\n                augmented = spectral_augment(\n                    cube,\n                    stable_rng(args.seed, stem, copy_index),\n                    args.spectral_gain,\n                    args.spectral_tilt,\n                    args.spectral_noise,\n                )\n                write_tiff(\n                    augmented,\n                    prepared / "images" / split / f"{suffix}.tiff",\n                    lo,\n                    hi,\n                    input_policy,\n                    pipeline_state,\n                )\n                write_label(prepared / "labels" / split / f"{suffix}.txt", rows)\n                train_images_written += 1\n\n    test_out = cache_root / "test_tiff"\n    if test_out.exists():\n        raise FileExistsError(f"Use a fresh workdir; keeping existing data: {test_out}")\n    test_out.mkdir(parents=True, exist_ok=True)\n    inference_geometry = {}\n    for stem, path in inference_by_stem.items():\n        inference_geometry[stem] = write_tiff(\n            read_cube(path),\n            test_out / f"{stem}.tiff",\n            lo,\n            hi,\n            input_policy,\n            pipeline_state,\n        )\n    (work / "inference_geometry.json").write_text(json.dumps(inference_geometry, indent=2))\n\n    data = {\n        "path": str(prepared),\n        "train": "images/train",\n        "val": "images/val",\n        "test": "images/test",\n        "channels": 16,\n        "names": {i: n for i, n in enumerate(names)},\n    }\n    yaml_path = work / "dataset.yaml"\n    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))\n\n    report = {\n        "train_images_total": len(stems),\n        "train_split": len(train_stems),\n        "val_split": len(val_stems),\n        "holdout_split": len(holdout_stems),\n        "prepared_train_images": train_images_written,\n        "competition_test_images": len(test_cubes),\n        "competition_ranking_images": len(ranking_cubes),\n        "class_names": names,\n        "excluded_ids": sorted(bad_stems),\n        "split_ids": split_map,\n        "split_mode": "manifest" if args.split_manifest else ("cv" if args.cv_fold_index is not None else "legacy"),\n        "cv_folds": args.cv_folds if args.cv_fold_index is not None else None,\n        "cv_fold_index": args.cv_fold_index,\n        "groups_source": groups_source,\n        "group_count": len(set(groups_by_stem.values())),\n        "input_policy": input_policy,\n        "spectral_augmentation": {\n            "copies": args.spectral_aug_copies,\n            "classes": args.spectral_aug_classes,\n            "gain_limit": args.spectral_gain,\n            "tilt_limit": args.spectral_tilt,\n            "noise_std": args.spectral_noise,\n        },\n        "class_counts": {split: np.bincount([int(row[0]) for stem in ids for row in boxes_by_stem[stem]], minlength=18).tolist() for split, ids in split_map.items()},\n        "band_low_0_5pct": lo.tolist() if lo is not None else None,\n        "band_high_99_5pct": hi.tolist() if hi is not None else None,\n        "normalization": normalization,\n        "spectral_preprocess": spectral_preprocess,\n        "spectral_pipeline_state": pipeline_state.to_dict() if pipeline_state is not None else None,\n    }\n    (work / "data_audit.json").write_text(json.dumps(report, indent=2))\n    return yaml_path, test_out, inference_geometry\n\n\ndef train(args, yaml_path):\n    model = model_class(args)(args.model)\n    result = model.train(\n        trainer=trainer_class(args),\n        data=str(yaml_path),\n        epochs=args.epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=args.run_name,\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=args.multi_scale,\n        patience=25,\n        close_mosaic=15,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=5.0,\n        translate=0.08,\n        scale=0.35,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.5,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef localization_finetune(args, yaml_path, checkpoint):\n    if args.localization_epochs <= 0:\n        return None\n    model = model_class(args)(str(checkpoint))\n    result = model.train(\n        data=str(yaml_path),\n        epochs=args.localization_epochs,\n        imgsz=args.imgsz,\n        batch=args.batch,\n        device=args.device,\n        workers=args.workers,\n        project=str(Path(args.workdir) / "runs"),\n        name=f"{args.run_name}_localize",\n        seed=args.seed,\n        deterministic=args.family == "yolo",\n        amp=args.family == "yolo",\n        rect=args.family == "yolo",\n        multi_scale=0.0,\n        patience=args.localization_epochs,\n        lr0=args.localization_lr0,\n        lrf=0.1,\n        cos_lr=True,\n        warmup_epochs=0.5,\n        close_mosaic=0,\n        hsv_h=0.0,\n        hsv_s=0.0,\n        hsv_v=0.0,\n        degrees=0.0,\n        translate=0.02,\n        scale=0.10,\n        fliplr=0.5,\n        flipud=0.0,\n        mosaic=0.0,\n        mixup=0.0,\n        copy_paste=0.0,\n        augmentations=[],\n        plots=False,\n        save=True,\n        verbose=True,\n        pretrained=True,\n    )\n    best = Path(result.save_dir) / "weights" / "best.pt"\n    if not best.exists():\n        raise FileNotFoundError(best)\n    return best\n\n\ndef predict(args, best, test_out, inference_geometry, tta):\n    model = model_class(args)(str(best))\n    results = model.predict(\n        source=str(test_out),\n        imgsz=args.imgsz,\n        conf=args.predict_conf,\n        iou=args.iou,\n        max_det=args.predict_max_det,\n        device=args.device,\n        augment=tta,\n        stream=True,\n        verbose=False,\n    )\n    rows = []\n    seen = set()\n    empty = []\n    for r in results:\n        seen.add(Path(r.path).stem)\n        if r.boxes is None or len(r.boxes) == 0:\n            empty.append(Path(r.path).stem)\n        stem = Path(r.path).stem\n        if r.boxes is None:\n            continue\n        xyxy = restore_boxes(\n            r.boxes.xyxy.detach().cpu().numpy(),\n            inference_geometry[stem],\n        )\n        confs = r.boxes.conf.detach().cpu().numpy()\n        classes = r.boxes.cls.detach().cpu().numpy().astype(int)\n        for box, score, cls in zip(xyxy, confs, classes):\n            x1, y1, x2, y2 = map(float, box)\n            if x2 <= x1 or y2 <= y1:\n                continue\n            rows.append(\n                {\n                    "image_id": image_id_value(stem),\n                    "class_id": int(cls),\n                    "confidence": float(score),\n                    "x1": x1,\n                    "y1": y1,\n                    "x2": x2,\n                    "y2": y2,\n                }\n            )\n\n    expected_ids = set(inference_geometry)\n    if seen != expected_ids:\n        raise RuntimeError("Some test images were not processed")\n    counts = pd.Series([row["image_id"] for row in rows]).value_counts()\n    (Path(args.workdir) / "prediction_audit.json").write_text(json.dumps({\n        "processed_images": len(seen),\n        "no_detection_ids": empty,\n        "prediction_confidence": args.predict_conf,\n        "prediction_max_det": args.predict_max_det,\n        "tta": tta,\n        "predictions_total": len(rows),\n        "predictions_per_image_min": int(counts.min()) if len(counts) else 0,\n        "predictions_per_image_median": float(counts.median()) if len(counts) else 0.0,\n        "predictions_per_image_max": int(counts.max()) if len(counts) else 0,\n    }, indent=2))\n    sub = pd.DataFrame(rows)\n    if sub.empty:\n        raise RuntimeError("No detections produced")\n    sub.insert(0, "id", np.arange(len(sub), dtype=int))\n    expected = ["id", "image_id", "class_id", "confidence", "x1", "y1", "x2", "y2"]\n    sub = sub[expected]\n    if args.sample_submission and pd.read_csv(args.sample_submission, nrows=0).columns.tolist() != expected:\n        raise ValueError("Sample submission columns do not match")\n    if not np.isfinite(sub[["confidence", "x1", "y1", "x2", "y2"]].to_numpy()).all():\n        raise ValueError("Non-finite submission values")\n    if not sub["id"].is_unique:\n        raise RuntimeError("Submission id is not unique")\n    if not sub["class_id"].between(0, 17).all():\n        raise RuntimeError("class_id outside 0..17")\n    if not sub["confidence"].between(0, 1).all():\n        raise RuntimeError("confidence outside 0..1")\n    if not ((sub["x2"] > sub["x1"]) & (sub["y2"] > sub["y1"])).all():\n        raise RuntimeError("Invalid boxes")\n    out_path = Path(args.workdir) / "submission.csv"\n    sub.to_csv(out_path, index=False)\n    return out_path\n\n\ndef unique_stems(files):\n    result = {}\n    for path in files:\n        if path.stem in result:\n            raise ValueError(f"Duplicate image or XML ID: {path.stem}")\n        result[path.stem] = path\n    return result\n\n\ndef model_class(args):\n    from ultralytics import RTDETR, YOLO\n\n    return RTDETR if args.family == "rtdetr" else YOLO\n\n\ndef metric_summary(metrics):\n    class_ids = [int(c) for c in metrics.box.ap_class_index]\n    per_class = {}\n    for position, class_id in enumerate(class_ids):\n        per_class[str(class_id)] = {\n            "ap50_95": float(metrics.box.ap[position]),\n            "ap50": float(metrics.box.ap50[position]),\n            "ap75": float(metrics.box.all_ap[position, 5]),\n            "precision": float(metrics.box.p[position]),\n            "recall": float(metrics.box.r[position]),\n        }\n    summary = {\n        "map50_95": float(metrics.box.map),\n        "map50": float(metrics.box.map50),\n        "map75": float(metrics.box.map75),\n        "localization_gap_map50_minus_map75": float(metrics.box.map50 - metrics.box.map75),\n        "per_class": per_class,\n        "worst_present_class_ap": min((row["ap50_95"] for row in per_class.values()), default=None),\n        "absent_class_ids": sorted(set(range(18)) - set(class_ids)),\n    }\n    matrix = np.asarray(metrics.confusion_matrix.matrix)\n    if matrix.shape == (19, 19):\n        true_positive = np.diag(matrix)[:18]\n        summary["false_positives_per_class"] = (\n            matrix[:18, :].sum(axis=1) - true_positive\n        ).astype(int).tolist()\n        summary["false_negatives_per_class"] = (\n            matrix[:, :18].sum(axis=0) - true_positive\n        ).astype(int).tolist()\n        summary["confusion_matrix_predicted_rows_true_columns"] = matrix.astype(int).tolist()\n    values = [summary["map50_95"], summary["map50"], summary["map75"]]\n    if not np.isfinite(values).all():\n        raise RuntimeError("Non-finite evaluation metrics")\n    return summary\n\n\ndef evaluate(args, best, yaml_path, splits, output_name, tta=False):\n    model = model_class(args)(str(best))\n    report = {\n        "checkpoint": str(best),\n        "sha256": hashlib.sha256(Path(best).read_bytes()).hexdigest(),\n        "tta": tta,\n    }\n    for split in splits:\n        metrics = model.val(data=str(yaml_path), split=split, imgsz=args.imgsz,\n                            batch=args.batch if args.batch > 0 else 4,\n                            device=args.device, workers=args.workers, conf=args.conf,\n                            max_det=args.max_det, iou=args.iou, plots=False,\n                            augment=tta,\n                            project=str(Path(args.workdir) / "evaluation"),\n                            name=f"{Path(output_name).stem}_{split}")\n        report[split] = metric_summary(metrics)\n        (Path(args.workdir) / output_name).write_text(json.dumps(report, indent=2))\n    return report\n\n\ndef promote_candidate(base_report, candidate_report, min_gain):\n    base = base_report["val"]\n    candidate = candidate_report["val"]\n    gain = candidate["map50_95"] - base["map50_95"]\n    map75_gain = candidate["map75"] - base["map75"]\n    worst_regression = candidate["worst_present_class_ap"] - base["worst_present_class_ap"]\n    passed = (gain >= min_gain or (gain >= 0 and map75_gain >= min_gain)) and worst_regression >= -0.03\n    return {\n        "passed": passed,\n        "map50_95_gain": gain,\n        "map75_gain": map75_gain,\n        "worst_present_class_ap_change": worst_regression,\n        "minimum_gain": min_gain,\n    }\n\n\ndef check_submission_gate(args, report):\n    holdout_map = report["test"]["map50_95"]\n    gate = {\n        "metric": "holdout_mAP50_95",\n        "value": holdout_map,\n        "minimum": args.min_holdout_map,\n        "passed": holdout_map >= args.min_holdout_map,\n    }\n    (Path(args.workdir) / "submission_gate.json").write_text(json.dumps(gate, indent=2))\n    return gate\n\n\ndef main():\n    p = argparse.ArgumentParser()\n    p.add_argument("--train-cubes", required=True)\n    p.add_argument("--test-cubes", required=True)\n    p.add_argument("--ranking-cubes")\n    p.add_argument("--train-xml", required=True)\n    p.add_argument("--class-file", required=True)\n    p.add_argument("--workdir", default="/kaggle/working/hsi_submission01")\n    p.add_argument("--cache-dir")\n    p.add_argument("--groups-file")\n    p.add_argument("--split-manifest")\n    p.add_argument("--cv-folds", type=int, default=3)\n    p.add_argument("--cv-fold-index", type=int)\n    p.add_argument("--model", default="yolo26m.pt")\n    p.add_argument("--checkpoint")\n    p.add_argument("--epochs", type=int, default=120)\n    p.add_argument("--imgsz", type=int, default=1024)\n    p.add_argument("--workers", type=int, default=2)\n    p.add_argument("--device", default="0")\n    p.add_argument("--seed", type=int, default=42)\n    p.add_argument("--input-policy", choices=["auto", "native", "square_pad"], default="auto")\n    p.add_argument("--stem-init", choices=["random", "rgb_mean"], default="rgb_mean")\n    p.add_argument("--normalization", choices=["N0", "N1", "N2", "N3", "N4"], default="N0")\n    p.add_argument("--spectral-preprocess", choices=["SP0", "SP1", "SP2", "SP3"], default="SP0")\n    p.add_argument("--normalization-samples-per-image", type=int, default=256)\n    p.add_argument("--mnf-components", type=int)\n    p.add_argument("--savgol-window", type=int, default=5)\n    p.add_argument("--savgol-order", type=int, default=2)\n    p.add_argument("--multi-scale", type=float, default=0.0)\n    p.add_argument("--spectral-aug-copies", type=int, default=0)\n    p.add_argument("--spectral-aug-classes", type=parse_class_ids, default=[])\n    p.add_argument("--spectral-gain", type=float, default=0.05)\n    p.add_argument("--spectral-tilt", type=float, default=0.03)\n    p.add_argument("--spectral-noise", type=float, default=0.01)\n    p.add_argument("--localization-epochs", type=int, default=0)\n    p.add_argument("--localization-lr0", type=float, default=0.0001)\n    p.add_argument("--promotion-min-gain", type=float, default=0.005)\n    p.add_argument("--tta", action="store_true")\n    p.add_argument("--tta-min-gain", type=float, default=0.002)\n    p.add_argument("--design-only", action="store_true")\n    p.add_argument("--conf", type=float, default=0.001)\n    p.add_argument("--predict-conf", type=float, default=0.001)\n    p.add_argument("--iou", type=float, default=0.65)\n    p.add_argument("--max-det", type=int, default=300)\n    p.add_argument("--predict-max-det", type=int, default=300)\n    p.add_argument("--min-holdout-map", type=float, default=0.50)\n    p.add_argument("--family", choices=["yolo", "rtdetr"], default="yolo")\n    p.add_argument("--batch", type=int, default=-1)\n    p.add_argument("--run-name", default="submission01")\n    p.add_argument("--sample-submission")\n    p.add_argument("--evaluate-holdout", action="store_true")\n    args = p.parse_args()\n    if args.spectral_aug_copies < 0:\n        raise ValueError("spectral_aug_copies must be non-negative")\n    if args.spectral_preprocess == "SP1" and args.mnf_components is None:\n        raise ValueError("SP1 requires --mnf-components selected on validation")\n    if not 0 <= args.multi_scale <= 0.9:\n        raise ValueError("multi_scale must be in 0..0.9")\n    if args.tta and args.family != "yolo":\n        raise ValueError("Built-in validated TTA is enabled only for YOLO in this runner")\n    import torch\n    if args.device != "cpu" and not torch.cuda.is_available():\n        raise RuntimeError("GPU is unavailable. Enable a Kaggle GPU and keep its CUDA PyTorch installation.")\n\n    random.seed(args.seed)\n    np.random.seed(args.seed)\n    Path(args.workdir).mkdir(parents=True, exist_ok=True)\n\n    versions = {name: importlib.metadata.version(name) for name in ("torch", "ultralytics", "numpy")}\n    run_config = {\n        "args": vars(args),\n        "resolved_input_policy": resolve_input_policy(args.family, args.input_policy),\n        "versions": versions,\n    }\n    (Path(args.workdir) / "run_config.json").write_text(json.dumps(run_config, indent=2))\n    yaml_path, test_out, inference_geometry = build_dataset(args)\n    if args.checkpoint:\n        base_checkpoint = Path(args.checkpoint)\n        if not base_checkpoint.exists():\n            raise FileNotFoundError(base_checkpoint)\n    else:\n        base_checkpoint = train(args, yaml_path)\n    base_report = evaluate(\n        args,\n        base_checkpoint,\n        yaml_path,\n        ["val"],\n        "evaluation_base.json",\n    )\n    selected_checkpoint = base_checkpoint\n    selected_report = base_report\n    selection = {"base_checkpoint": str(base_checkpoint), "localization_finetune": None, "tta": None}\n    finetuned_checkpoint = localization_finetune(args, yaml_path, base_checkpoint)\n    if finetuned_checkpoint:\n        finetuned_report = evaluate(\n            args,\n            finetuned_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_localized.json",\n        )\n        decision = promote_candidate(base_report, finetuned_report, args.promotion_min_gain)\n        decision["checkpoint"] = str(finetuned_checkpoint)\n        selection["localization_finetune"] = decision\n        if decision["passed"]:\n            selected_checkpoint = finetuned_checkpoint\n            selected_report = finetuned_report\n    selected_tta = False\n    if args.tta:\n        tta_report = evaluate(\n            args,\n            selected_checkpoint,\n            yaml_path,\n            ["val"],\n            "evaluation_tta.json",\n            tta=True,\n        )\n        decision = promote_candidate(selected_report, tta_report, args.tta_min_gain)\n        selection["tta"] = decision\n        if decision["passed"]:\n            selected_report = tta_report\n            selected_tta = True\n    selection["selected_checkpoint"] = str(selected_checkpoint)\n    selection["selected_checkpoint_sha256"] = hashlib.sha256(Path(selected_checkpoint).read_bytes()).hexdigest()\n    selection["selected_tta"] = selected_tta\n    (Path(args.workdir) / "selection.json").write_text(json.dumps(selection, indent=2))\n    if args.design_only:\n        print(selection)\n        print("Design-only run complete. Holdout and competition test prediction were not used.")\n        return\n    holdout_report = evaluate(\n        args,\n        selected_checkpoint,\n        yaml_path,\n        ["test"],\n        "evaluation_holdout.json",\n        tta=selected_tta,\n    )\n    report = {\n        "checkpoint": str(selected_checkpoint),\n        "sha256": selection["selected_checkpoint_sha256"],\n        "tta": selected_tta,\n        "val": selected_report["val"],\n        "test": holdout_report["test"],\n    }\n    (Path(args.workdir) / "evaluation.json").write_text(json.dumps(report, indent=2))\n    gate = check_submission_gate(args, report)\n    print(gate)\n    if not gate["passed"]:\n        print("Submission blocked: holdout mAP50-95 is below minimum.")\n        print(selected_checkpoint)\n        return\n    submission = predict(args, selected_checkpoint, test_out, inference_geometry, selected_tta)\n    print(selected_checkpoint)\n    print(submission)\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/build_folds.py').write_text('import argparse\nimport csv\nimport hashlib\nimport json\nfrom collections import Counter, defaultdict\nfrom pathlib import Path\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\n\n\nEXCLUDED_IDS = ("1227", "1836", "1855")\nAREA_EDGES = (0, 32**2, 96**2)\n\n\ndef digest_file(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef digest_json(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()\n\n\ndef unique_files(root, suffixes):\n    result = {}\n    for path in sorted(Path(root).rglob("*")):\n        if path.is_file() and path.suffix.lower() in suffixes:\n            if path.stem in result:\n                raise ValueError(f"Duplicate image ID {path.stem}: {result[path.stem]}, {path}")\n            result[path.stem] = path\n    if not result:\n        raise ValueError(f"No matching files in {root}")\n    return result\n\n\ndef non_object_metadata(root):\n    result = defaultdict(list)\n\n    def visit(node, prefix):\n        if node.tag.lower() == "object":\n            return\n        key = f"{prefix}/{node.tag}" if prefix else node.tag\n        result[key].append({"text": (node.text or "").strip(), "attributes": dict(node.attrib)})\n        for child in node:\n            visit(child, key)\n\n    visit(root, "")\n    return dict(result)\n\n\ndef read_annotation(path, names):\n    root = ET.parse(path).getroot()\n    width = float(root.findtext("size/width"))\n    height = float(root.findtext("size/height"))\n    if not np.isfinite([width, height]).all() or min(width, height) <= 0:\n        raise ValueError(f"Invalid original dimensions in {path}")\n    counts = np.zeros(len(names), dtype=np.int64)\n    areas = []\n    class_sizes = np.zeros((len(names), 3), dtype=np.int64)\n    clipped = discarded = 0\n    for obj in root.findall("object"):\n        name = (obj.findtext("name") or "").strip()\n        if name not in names:\n            raise ValueError(f"Unknown class {name!r} in {path}")\n        box = np.array([float(obj.findtext(f"bndbox/{key}")) for key in ("xmin", "ymin", "xmax", "ymax")])\n        if not np.isfinite(box).all():\n            raise ValueError(f"Non-finite box in {path}")\n        bounded = np.clip(box, 0, [width, height, width, height])\n        clipped += int(not np.array_equal(box, bounded))\n        x1, y1, x2, y2 = bounded\n        if x2 <= x1 or y2 <= y1:\n            discarded += 1\n            continue\n        area = float((x2 - x1) * (y2 - y1))\n        class_id = names.index(name)\n        counts[class_id] += 1\n        areas.append(area)\n        class_sizes[class_id, int(np.searchsorted(AREA_EDGES[1:], area, side="right"))] += 1\n    return {\n        "width": width, "height": height, "class_counts": counts.tolist(),\n        "class_presence": (counts > 0).astype(int).tolist(), "box_areas": areas,\n        "class_size_counts": class_sizes.tolist(), "size_counts": class_sizes.sum(0).tolist(),\n        "clipped_box_count": clipped, "discarded_box_count": discarded,\n        "metadata": non_object_metadata(root),\n    }\n\n\ndef load_groups(ids, groups_file=None, groups_evidence=None):\n    if groups_file is None:\n        if groups_evidence is not None:\n            raise ValueError("--groups-evidence requires --groups-file")\n        return {stem: stem for stem in ids}, {\n            "mode": "image_id_only",\n            "relationship_origin": "identity_fallback",\n            "source_grouping_verified": False,\n            "verification_status": "no_source_relationship_claimed",\n        }\n    if groups_evidence is None or not Path(groups_evidence).read_text().strip():\n        raise ValueError("A group mapping requires a non-empty --groups-evidence file documenting the relationship")\n    with Path(groups_file).open(newline="") as handle:\n        reader = csv.DictReader(handle)\n        if set(reader.fieldnames or []) != {"image_id", "group_id"}:\n            raise ValueError("Groups CSV must contain exactly image_id,group_id")\n        groups = {}\n        for row in reader:\n            stem, group = row["image_id"].strip(), row["group_id"].strip()\n            if not stem or not group or stem in groups:\n                raise ValueError("Empty or duplicate image ID/group in groups CSV")\n            groups[stem] = group\n    if set(ids) - groups.keys():\n        raise ValueError(f"Groups CSV missing IDs: {sorted(set(ids) - groups.keys())[:10]}")\n    return {stem: groups[stem] for stem in ids}, {\n        "mode": "provided_group_mapping",\n        "relationship_origin": "caller_provided_csv",\n        "source_grouping_verified": False,\n        "verification_status": "requires_evidence_review",\n        "groups_file_sha256": digest_file(groups_file),\n        "groups_evidence_sha256": digest_file(groups_evidence),\n        "groups_evidence": Path(groups_evidence).read_text(),\n        "unused_group_ids": sorted(set(groups) - set(ids)),\n    }\n\n\ndef check_split(split, ids, groups):\n    flat = [stem for part in ("train", "val", "test") for stem in split[part]]\n    if len(flat) != len(set(flat)) or set(flat) != set(ids):\n        raise ValueError("Split must cover every usable image exactly once")\n    seen = {}\n    for part in ("train", "val", "test"):\n        if not split[part]:\n            raise ValueError(f"Empty {part} partition")\n        for stem in split[part]:\n            group = groups[stem]\n            if group in seen and seen[group] != part:\n                raise ValueError(f"Group {group} leaks across partitions")\n            seen[group] = part\n\n\ndef assign_groups(ids, groups, records, seed):\n    members = defaultdict(list)\n    for stem in sorted(ids):\n        members[groups[stem]].append(stem)\n    if len(members) < 3:\n        raise ValueError("Three-fold CV requires at least three development groups")\n    keys = sorted(members)\n    vectors = np.asarray([\n        np.sum([records[s]["class_presence"] + records[s]["class_counts"] + records[s]["size_counts"] + [1] for s in members[g]], axis=0).tolist() + [1]\n        for g in keys\n    ], dtype=float)\n    total = vectors.sum(0)\n    active = total > 0\n    normalized = vectors[:, active] / total[active]\n    rng = np.random.default_rng(seed)\n    ties = rng.random(len(keys))\n    order = sorted(range(len(keys)), key=lambda i: (-float(np.max(normalized[i])), -float(np.sum(normalized[i])), ties[i]))\n    loads = np.zeros((3, normalized.shape[1]))\n    counts = np.zeros(3, dtype=int)\n    chosen = {}\n    priority = rng.permutation(3).tolist()\n    for i in order:\n        scores = []\n        for fold in range(3):\n            trial = loads.copy()\n            trial[fold] += normalized[i]\n            scores.append(float(np.square(trial - trial.mean(0)).sum()))\n        fold = min(range(3), key=lambda f: (scores[f], counts[f], priority.index(f)))\n        loads[fold] += normalized[i]\n        counts[fold] += len(members[keys[i]])\n        chosen[keys[i]] = fold\n    return {stem: chosen[groups[stem]] for stem in ids}\n\n\ndef summarize(ids, records, groups, names):\n    counts = np.sum([records[s]["class_counts"] for s in ids], axis=0).astype(int)\n    presence = np.sum([records[s]["class_presence"] for s in ids], axis=0).astype(int)\n    areas = [area for s in ids for area in records[s]["box_areas"]]\n    histogram, _ = np.histogram(areas, bins=[0, 256, 1024, 4096, 9216, 16384, 65536, np.inf])\n    return {\n        "image_count": len(ids), "group_count": len({groups[s] for s in ids}),\n        "object_count": int(counts.sum()), "class_object_counts": dict(zip(names, counts.tolist())),\n        "class_image_counts": dict(zip(names, presence.tolist())),\n        "missing_classes": [name for name, count in zip(names, counts) if count == 0],\n        "size_counts": dict(zip(("small", "medium", "large"), np.sum([records[s]["size_counts"] for s in ids], axis=0).astype(int).tolist())),\n        "class_size_counts": dict(zip(names, np.sum([records[s]["class_size_counts"] for s in ids], axis=0).astype(int).tolist())),\n        "box_area_histogram": {"edges": [0, 256, 1024, 4096, 9216, 16384, 65536, "infinity"], "counts": histogram.tolist()},\n    }\n\n\ndef build_folds(xml_dir, cube_dir, class_file, legacy_split, output_dir, groups_file=None, groups_evidence=None, seed=42):\n    names = [line.strip() for line in Path(class_file).read_text().splitlines() if line.strip()]\n    if not names or len(names) != len(set(names)):\n        raise ValueError("Class names must be nonempty and unique")\n    xmls = unique_files(xml_dir, {".xml"})\n    cubes = unique_files(cube_dir, {".png", ".npy", ".npz", ".tif", ".tiff"})\n    if set(xmls) != set(cubes):\n        raise ValueError("Annotation and cube image IDs differ")\n    ids = sorted(set(xmls) - set(EXCLUDED_IDS))\n    legacy = json.loads(Path(legacy_split).read_text())\n    if set(legacy) != {"train", "val", "test"}:\n        raise ValueError("Legacy split requires train, val, test")\n    legacy = {key: list(map(str, value)) for key, value in legacy.items()}\n    check_split(legacy, ids, {stem: stem for stem in ids})\n    records = {stem: read_annotation(xmls[stem], names) for stem in ids}\n    groups, group_info = load_groups(ids, groups_file, groups_evidence)\n    holdout_groups = {groups[stem] for stem in legacy["test"]}\n    holdout = sorted(stem for stem in ids if groups[stem] in holdout_groups)\n    development = sorted(set(ids) - set(holdout))\n    assignments = assign_groups(development, groups, records, seed)\n    folds = []\n    for index in range(3):\n        split = {"train": [s for s in development if assignments[s] != index], "val": [s for s in development if assignments[s] == index], "test": holdout}\n        check_split(split, ids, groups)\n        folds.append(split)\n    if sorted(s for split in folds for s in split["val"]) != development:\n        raise AssertionError("Development images must validate exactly once")\n    fields = defaultdict(Counter)\n    inventory = {}\n    for stem, record in records.items():\n        for key, values in record["metadata"].items():\n            fields[key].update(json.dumps(v, sort_keys=True) for v in values)\n        inventory[stem] = {\n            "xml": str(xmls[stem].relative_to(xml_dir)), "cube": str(cubes[stem].relative_to(cube_dir)),\n            "xml_sha256": digest_file(xmls[stem]), "cube_size_bytes": cubes[stem].stat().st_size,\n            "xml_metadata": record["metadata"], "original_width": record["width"], "original_height": record["height"],\n            "original_box_areas": record["box_areas"], "clipped_box_count": record["clipped_box_count"],\n            "discarded_box_count": record["discarded_box_count"],\n        }\n    grouping_fields = [key for key in fields if any(word in key.lower() for word in ("capture", "session", "source", "folder", "sequence", "scene", "video"))]\n    candidate_fields = [key for key in grouping_fields if len(fields[key]) > 1]\n    non_informative_fields = [key for key in grouping_fields if len(fields[key]) <= 1]\n    config = {"seed": seed, "cv_folds": 3, "excluded_ids": list(EXCLUDED_IDS), "area_bins": list(AREA_EDGES),\n              "legacy_split_sha256": digest_file(legacy_split), "class_file_sha256": digest_file(class_file),\n              "group_mapping_sha256": digest_json(groups), "annotation_inventory_sha256": digest_json(inventory),\n              "groups_evidence_sha256": group_info.get("groups_evidence_sha256"),\n              "algorithm": "seeded_greedy_multilabel_presence_object_counts_original_area_bins_image_and_group_counts"}\n    audit = {\n        "status": "folds_built_not_training_results", "code_hash": digest_file(__file__), "config": config,\n        "config_hash": digest_json(config), "groups": group_info,\n        "excluded_ids": {stem: {"present": stem in xmls,\n                                "reason": "Existing runner exclusion retained for comparability; the historical mismatch rationale was not re-verified",\n                                "evidence_status": "historical_policy_unverified"} for stem in EXCLUDED_IDS},\n        "holdout": {"status": "previously_inspected", "legacy_count": len(legacy["test"]), "group_closed_count": len(holdout),\n                    "added_for_group_closure": sorted(set(holdout) - set(legacy["test"])),\n                    "previously_trained_by_legacy_checkpoint": sorted(set(holdout) & set(legacy["train"])),\n                    "legacy_checkpoint_declared_group_overlap": sorted(holdout_groups & {groups[s] for s in legacy["train"]}),\n                    "use": "Freeze new designs before evaluation; cannot claim an untouched final holdout. Existing checkpoints cannot provide new CV evidence."},\n        "metadata_audit": {"non_object_fields": {key: {"unique_values": len(values), "value_counts": dict(sorted(values.items()))} for key, values in sorted(fields.items())},\n                           "candidate_group_fields": candidate_fields,\n                           "non_informative_group_fields": non_informative_fields,\n                           "unresolved_metadata_candidates": candidate_fields if groups_file is None else [],\n                           "xml_folders": dict(Counter(str(path.parent.relative_to(xml_dir)) for path in xmls.values())),\n                           "cube_folders": dict(Counter(str(path.parent.relative_to(cube_dir)) for path in cubes.values())),\n                           "numeric_filename_count": sum(stem.isdigit() for stem in ids),\n                           "numeric_name_policy": "Numbers alone are not capture/session evidence; no numeric or visual grouping inferred",\n                           "conclusion": ("No varying official capture/session/source field exists; image_id_only is the supported fallback" if not candidate_fields else "Only image identities are currently supported; varying source/session candidates require provenance review") if groups_file is None else "Caller-provided mapping used for isolation; no source relationship is claimed until attached provenance is reviewed"},\n        "coordinate_policy": "Original XML pixel coordinates; clip to XML image bounds; discard nonpositive boxes as existing runner does; no resize before areas",\n        "assertions": {"image_disjoint": True, "declared_group_mapping_disjoint": True,\n                       "verified_source_group_disjoint": True if group_info["source_grouping_verified"] else None,\n                       "all_usable_ids_covered": True, "validation_once_per_development_id": True,\n                       "legacy_holdout_preserved": True},\n        "folds": [{"fold": i, "split_hash": digest_json(split), "partitions": {part: summarize(stems, records, groups, names) for part, stems in split.items()}} for i, split in enumerate(folds)],\n    }\n    out = Path(output_dir)\n    out.mkdir(parents=True, exist_ok=True)\n    paths = [out / f"fold{i}.json" for i in range(3)] + [out / "groups.csv", out / "fold_audit.json", out / "metadata_inventory.json"]\n    if any(path.exists() for path in paths):\n        raise FileExistsError("Use a fresh output directory; frozen fold artifacts cannot be overwritten")\n    for i, split in enumerate(folds):\n        (out / f"fold{i}.json").write_text(json.dumps(split, indent=2) + "\\n")\n    with (out / "groups.csv").open("w", newline="") as handle:\n        writer = csv.writer(handle)\n        writer.writerow(["image_id", "group_id"])\n        writer.writerows((stem, groups[stem]) for stem in ids)\n    (out / "fold_audit.json").write_text(json.dumps(audit, indent=2, allow_nan=False) + "\\n")\n    (out / "metadata_inventory.json").write_text(json.dumps(inventory, indent=2, allow_nan=False) + "\\n")\n    return audit\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--xml-dir", required=True)\n    parser.add_argument("--cube-dir", required=True)\n    parser.add_argument("--class-file", required=True)\n    parser.add_argument("--legacy-split", required=True)\n    parser.add_argument("--output-dir", required=True)\n    parser.add_argument("--groups-file")\n    parser.add_argument("--groups-evidence")\n    parser.add_argument("--seed", type=int, default=42)\n    audit = build_folds(**vars(parser.parse_args()))\n    print(json.dumps({"status": audit["status"], "grouping": audit["groups"]["mode"], "fold_sizes": [{part: value["image_count"] for part, value in fold["partitions"].items()} for fold in audit["folds"]], "holdout": audit["holdout"]["status"]}, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/evaluate_size_ap.py').write_text('import argparse\nimport contextlib\nimport copy\nimport importlib.metadata\nimport io\nimport json\nimport math\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\n\n\nAREA_LABELS = ("all", "small", "medium", "large")\nOFFICIAL_SOURCE = "https://github.com/cocodataset/cocoapi/blob/master/PythonAPI/pycocotools/cocoeval.py"\n\n\ndef _prepare_inputs(coco_gt, predictions):\n    gt = copy.deepcopy(coco_gt)\n    predictions = copy.deepcopy(list(predictions))\n    for key in ("images", "categories", "annotations"):\n        if key not in gt:\n            raise ValueError(f"COCO ground truth is missing {key}")\n    image_ids = [image["id"] for image in gt["images"]]\n    category_ids = [category["id"] for category in gt["categories"]]\n    if not image_ids or not category_ids:\n        raise ValueError("At least one image and category are required")\n    if len(set(image_ids)) != len(image_ids) or len(set(category_ids)) != len(category_ids):\n        raise ValueError("Image and category IDs must be unique")\n    if any(not isinstance(i, int) or isinstance(i, bool) for i in image_ids + category_ids):\n        raise ValueError("Image and category IDs must be integers")\n    for image in gt["images"]:\n        if any(not math.isfinite(float(image.get(k, 0))) or image.get(k, 0) <= 0 for k in ("width", "height")):\n            raise ValueError("Original image width and height must be finite and positive")\n    supplied_areas_replaced = 0\n    for is_prediction, rows in ((False, gt["annotations"]), (True, predictions)):\n        for index, row in enumerate(rows, 1):\n            if row["image_id"] not in image_ids or row["category_id"] not in category_ids:\n                raise ValueError("Annotation or prediction references an unknown image/category")\n            bbox = np.asarray(row.get("bbox", []), dtype=np.float64)\n            if bbox.shape != (4,) or not np.isfinite(bbox).all() or (bbox[2:] <= 0).any():\n                raise ValueError("Bboxes must be finite original-pixel XYWH with positive width/height")\n            area = float(bbox[2] * bbox[3])\n            if not math.isfinite(area):\n                raise ValueError("BBox area must be finite")\n            row["bbox"] = bbox.tolist()\n            if is_prediction:\n                score = float(row["score"])\n                if not math.isfinite(score) or not 0 <= score <= 1:\n                    raise ValueError("Prediction scores must be finite and between zero and one")\n                row["score"] = score\n            else:\n                supplied_areas_replaced += int("area" in row and row["area"] != area)\n                row["area"] = area\n                row["id"] = index\n                row["iscrowd"] = int(bool(row.get("iscrowd", 0)))\n                if row.get("ignore", 0) and not row["iscrowd"]:\n                    raise ValueError("Non-crowd ignore annotations need an explicit scoring policy")\n    gt.setdefault("info", {})\n    return gt, predictions, supplied_areas_replaced\n\n\ndef _mean_available(values):\n    values = np.asarray(values)\n    available = values[values > -1]\n    return float(available.mean()) if available.size else None\n\n\ndef _metric_row(evaluator, area_index, class_index=None):\n    precision = evaluator.eval["precision"][:, :, :, area_index, -1]\n    recall = evaluator.eval["recall"][:, :, area_index, -1]\n    if class_index is not None:\n        precision = precision[:, :, class_index]\n        recall = recall[:, class_index]\n    thresholds = evaluator.params.iouThrs\n    return {\n        "AP50_95": _mean_available(precision),\n        "AP50": _mean_available(precision[np.isclose(thresholds, 0.5)]),\n        "AP75": _mean_available(precision[np.isclose(thresholds, 0.75)]),\n        "AR50_95": _mean_available(recall),\n        "recall50": _mean_available(recall[np.isclose(thresholds, 0.5)]),\n        "recall75": _mean_available(recall[np.isclose(thresholds, 0.75)]),\n    }\n\n\ndef _evaluate(coco_gt, coco_dt, *, strict, max_dets):\n    from pycocotools.cocoeval import COCOeval\n\n    evaluator = COCOeval(coco_gt, coco_dt, "bbox")\n    evaluator.params.maxDets = sorted(set((1, min(10, max_dets), max_dets)))\n    if strict:\n        evaluator.params.areaRng = [\n            [0, float("inf")],\n            [0, float(np.nextafter(1024.0, -np.inf))],\n            [1024, float(np.nextafter(9216.0, -np.inf))],\n            [9216, float("inf")],\n        ]\n    evaluator.evaluate()\n    evaluator.accumulate()\n    categories = {category["id"]: category["name"] for category in coco_gt.dataset["categories"]}\n    return {\n        "size_metrics": {label: _metric_row(evaluator, i) for i, label in enumerate(AREA_LABELS)},\n        "class_by_size": [\n            {"category_id": int(category_id), "category_name": categories[category_id],\n             "sizes": {label: _metric_row(evaluator, i, k) for i, label in enumerate(AREA_LABELS)}}\n            for k, category_id in enumerate(evaluator.params.catIds)\n        ],\n    }\n\n\ndef evaluate_size_ap(coco_gt, predictions, *, max_dets=100):\n    from pycocotools.coco import COCO\n\n    if not isinstance(max_dets, int) or isinstance(max_dets, bool) or max_dets < 1:\n        raise ValueError("max_dets must be a positive integer")\n    gt, predictions, replaced = _prepare_inputs(coco_gt, predictions)\n    with contextlib.redirect_stdout(io.StringIO()):\n        ground_truth = COCO()\n        ground_truth.dataset = gt\n        ground_truth.createIndex()\n        if predictions:\n            detections = ground_truth.loadRes(predictions)\n        else:\n            detections = COCO()\n            detections.dataset = {"images": copy.deepcopy(gt["images"]),\n                                  "categories": copy.deepcopy(gt["categories"]), "annotations": []}\n            detections.createIndex()\n        standard = _evaluate(ground_truth, detections, strict=False, max_dets=max_dets)\n        strict = _evaluate(ground_truth, detections, strict=True, max_dets=max_dets)\n    raw_areas = [{"annotation_id": row["id"], "image_id": row["image_id"],\n                  "category_id": row["category_id"], "area": row["area"], "iscrowd": row["iscrowd"]}\n                 for row in gt["annotations"]]\n    bins = [0, 64, 256, 1024, 4096, 9216, 16384, 65536, float("inf")]\n    histogram = []\n    for lower, upper in zip(bins[:-1], bins[1:]):\n        rows = [row for row in raw_areas if lower <= row["area"] < upper]\n        histogram.append({"lower_inclusive": lower, "upper_exclusive": upper if math.isfinite(upper) else None,\n                          "count": len(rows), "non_crowd_count": sum(not row["iscrowd"] for row in rows)})\n    return {\n        "coordinate_space": "original_image_pixels_xywh",\n        "ground_truth_area_policy": "bbox_width_times_height_without_rescaling",\n        "supplied_gt_areas_replaced": replaced,\n        "pycocotools_version": importlib.metadata.version("pycocotools"),\n        "evaluator_source": OFFICIAL_SOURCE,\n        "iou_thresholds": [round(0.5 + 0.05 * i, 2) for i in range(10)],\n        "max_detections_per_image_per_class": max_dets,\n        "confidence_filter": None,\n        "boundary_policy": {\n            "roadmap_strict": "small [0,1024), medium [1024,9216), large [9216,infinity)",\n            "coco_standard": "small [0,1024], medium [1024,9216], large [9216,1e10]; inclusive overlapping boundaries",\n        },\n        "roadmap_strict": strict,\n        "coco_standard": standard,\n        "raw_box_areas": raw_areas,\n        "box_area_histogram": histogram,\n    }\n\n\ndef _iou(first, second):\n    x = max(first[0], second[0])\n    y = max(first[1], second[1])\n    right = min(first[0] + first[2], second[0] + second[2])\n    bottom = min(first[1] + first[3], second[1] + second[3])\n    intersection = max(0.0, right - x) * max(0.0, bottom - y)\n    return intersection / (first[2] * first[3] + second[2] * second[3] - intersection)\n\n\ndef detection_confusion(coco_gt, predictions, *, confidence_threshold, iou_threshold):\n    if not math.isfinite(confidence_threshold) or not 0 <= confidence_threshold <= 1:\n        raise ValueError("confidence_threshold must be between zero and one")\n    if not math.isfinite(iou_threshold) or not 0 < iou_threshold <= 1:\n        raise ValueError("iou_threshold must be greater than zero and at most one")\n    gt, predictions, _ = _prepare_inputs(coco_gt, predictions)\n    if any(row["iscrowd"] for row in gt["annotations"]):\n        raise ValueError("This independent confusion diagnostic does not support crowd annotations")\n    categories = sorted(gt["categories"], key=lambda category: category["id"])\n    indexes = {category["id"]: i for i, category in enumerate(categories)}\n    background = len(categories)\n    matrix = np.zeros((background + 1, background + 1), dtype=np.int64)\n    truths_by_image = defaultdict(list)\n    predictions_by_image = defaultdict(list)\n    for row in gt["annotations"]:\n        truths_by_image[row["image_id"]].append(row)\n    for row in predictions:\n        if row["score"] >= confidence_threshold:\n            predictions_by_image[row["image_id"]].append(row)\n    for image in gt["images"]:\n        truths = truths_by_image[image["id"]]\n        unmatched = set(range(len(truths)))\n        for prediction in sorted(predictions_by_image[image["id"]], key=lambda row: -row["score"]):\n            matches = [(i, _iou(prediction["bbox"], truths[i]["bbox"])) for i in sorted(unmatched)]\n            best = max(matches, key=lambda match: match[1]) if matches else None\n            predicted_class = indexes[prediction["category_id"]]\n            if best is not None and best[1] >= iou_threshold:\n                true_index = best[0]\n                unmatched.remove(true_index)\n                matrix[indexes[truths[true_index]["category_id"]], predicted_class] += 1\n            else:\n                matrix[background, predicted_class] += 1\n        for index in unmatched:\n            matrix[indexes[truths[index]["category_id"]], background] += 1\n    per_class = []\n    for index, category in enumerate(categories):\n        true_positive = int(matrix[index, index])\n        predicted_count = int(matrix[:, index].sum())\n        true_count = int(matrix[index, :].sum())\n        per_class.append({"category_id": category["id"], "category_name": category["name"],\n                          "true_positives": true_positive, "false_positives": predicted_count - true_positive,\n                          "false_negatives": true_count - true_positive,\n                          "precision": true_positive / predicted_count if predicted_count else None,\n                          "recall": true_positive / true_count if true_count else None})\n    return {"matrix": matrix.tolist(), "orientation": "rows=true, columns=predicted",\n            "category_ids": [category["id"] for category in categories] + [None],\n            "labels": [category["name"] for category in categories] + ["background"],\n            "confidence_threshold": confidence_threshold, "iou_threshold": iou_threshold,\n            "matching_policy": "descending score, class-agnostic highest-IoU unmatched GT; stable input-order ties",\n            "per_class": per_class}\n\n\ndef main():\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--ground-truth", required=True, type=Path)\n    parser.add_argument("--predictions", required=True, type=Path)\n    parser.add_argument("--output", required=True, type=Path)\n    parser.add_argument("--max-dets", type=int, default=100)\n    parser.add_argument("--confusion-confidence", type=float)\n    parser.add_argument("--confusion-iou", type=float)\n    args = parser.parse_args()\n    if (args.confusion_confidence is None) != (args.confusion_iou is None):\n        parser.error("Provide both confusion thresholds or neither")\n    gt = json.loads(args.ground_truth.read_text())\n    predictions = json.loads(args.predictions.read_text())\n    report = evaluate_size_ap(gt, predictions, max_dets=args.max_dets)\n    if args.confusion_confidence is not None:\n        report["confusion"] = detection_confusion(gt, predictions, confidence_threshold=args.confusion_confidence,\n                                                  iou_threshold=args.confusion_iou)\n    args.output.parent.mkdir(parents=True, exist_ok=True)\n    args.output.write_text(json.dumps(report, indent=2, allow_nan=False) + "\\n")\n\n\nif __name__ == "__main__":\n    main()\n')
Path('/kaggle/working/run_day1_diagnostics.py').write_text('from pathlib import Path\nfrom datetime import datetime, timezone\nimport hashlib\nimport importlib.metadata\nimport json\nimport subprocess\nimport sys\nimport time\nimport traceback\nimport zipfile\n\nimport numpy as np\nimport kagglehub\nimport torch\n\nimport hsi_runner as runner\nfrom evaluate_size_ap import evaluate_size_ap, detection_confusion\n\n\nOUT = Path("/kaggle/working/roadmap_day1")\nOUT.mkdir(parents=True, exist_ok=True)\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1048576), b""):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef save(name, value):\n    (OUT / name).write_text(json.dumps(value, indent=2, allow_nan=False))\n\n\ndef event(experiment_id, status, **fields):\n    row = dict(experiment_id=experiment_id, status=status,\n               recorded_at=datetime.now(timezone.utc).isoformat(), **fields)\n    with (OUT / "experiment_ledger.jsonl").open("a") as stream:\n        stream.write(json.dumps(row, allow_nan=False) + "\\n")\n    print(json.dumps(row, allow_nan=False), flush=True)\n\n\ndef one(root, pattern):\n    matches = list(root.rglob(pattern))\n    if len(matches) != 1:\n        raise RuntimeError(f"Expected one {pattern} in {root}, found {len(matches)}")\n    return matches[0]\n\n\ndef diagnostics(base, model_id, artifact, checkpoint_hash, family, run_dir):\n    started = time.monotonic()\n    work = one(artifact, run_dir)\n    weights = one(work, "best.pt")\n    if sha(weights) != checkpoint_hash:\n        raise RuntimeError(f"Historical checkpoint hash mismatch: {model_id}")\n    split = json.loads((work / "split_manifest.json").read_text())\n    audit = json.loads((work / "data_audit.json").read_text())\n    config = json.loads((work / "run_config.json").read_text())\n    names = runner.read_classes(base / "class.txt")\n    cube_paths = runner.unique_stems(runner.collect_cubes(base / "data_train/data_train/VIS"))\n    xml_paths = runner.annotation_map(base / "data_train/data_train/Annotations/VIS")\n    lo = np.asarray(audit["band_low_0_5pct"], dtype=np.float32)\n    hi = np.asarray(audit["band_high_99_5pct"], dtype=np.float32)\n    policy = config.get("resolved_input_policy", "native")\n    if policy not in ("native", "square_pad"):\n        raise ValueError(policy)\n    prepared = Path("/kaggle/temp/roadmap_day1") / model_id\n    prepared.mkdir(parents=True, exist_ok=False)\n    images, annotations, geometries = [], [], {}\n    for stem in split["val"]:\n        cube = runner.read_cube(cube_paths[stem])\n        width, height, rows = runner.parse_voc(xml_paths[stem], {name: i for i, name in enumerate(names)})\n        if cube.shape[:2] != (height, width):\n            raise ValueError(f"Spatial mismatch: {stem}")\n        images.append({"id": int(stem), "width": width, "height": height})\n        for cid, cx, cy, bw, bh in rows:\n            bbox = [(cx - bw / 2) * width, (cy - bh / 2) * height, bw * width, bh * height]\n            annotations.append({"id": len(annotations) + 1, "image_id": int(stem), "category_id": int(cid), "bbox": bbox, "area": bbox[2] * bbox[3], "iscrowd": 0})\n        geometries[stem] = runner.write_tiff(cube, prepared / f"{stem}.tiff", lo, hi, policy)\n    gt = {"images": images, "annotations": annotations, "categories": [{"id": i, "name": n} for i, n in enumerate(names)]}\n    save(f"{model_id}_gt_original.json", gt)\n    from ultralytics import RTDETR, YOLO\n    model = (RTDETR if family == "rtdetr" else YOLO)(str(weights))\n    device = "0" if torch.cuda.is_available() else "cpu"\n    if device == "0":\n        torch.ones(1, device="cuda").add_(1)\n        torch.cuda.synchronize()\n    predictions, seen = [], set()\n    for result in model.predict(source=str(prepared), imgsz=config["args"]["imgsz"], conf=0.001,\n                                iou=0.65, max_det=300, device=device, stream=True, verbose=False):\n        stem = Path(result.path).stem\n        seen.add(stem)\n        boxes = runner.restore_boxes(result.boxes.xyxy.cpu().numpy(), geometries[stem])\n        for box, score, cid in zip(boxes, result.boxes.conf.cpu().numpy(), result.boxes.cls.cpu().numpy()):\n            x1, y1, x2, y2 = map(float, box)\n            if x2 > x1 and y2 > y1:\n                predictions.append({"image_id": int(stem), "category_id": int(cid), "bbox": [x1, y1, x2 - x1, y2 - y1], "score": float(score)})\n    if seen != set(split["val"]):\n        raise RuntimeError("Incomplete validation inference")\n    save(f"{model_id}_predictions_original.json", predictions)\n    report = evaluate_size_ap(gt, predictions, max_dets=300)\n    report["confusion_at_fixed_diagnostic_threshold"] = detection_confusion(gt, predictions, confidence_threshold=0.25, iou_threshold=0.5)\n    report["threshold_note"] = "0.25/0.5 is a fixed reporting operating point, not an annotation-candidate or submission threshold."\n    report["provenance"] = {"checkpoint_hash": checkpoint_hash, "config_hash": sha(work / "run_config.json"),\n                            "split_hash": sha(work / "split_manifest.json"), "code_hash": sha(__file__),\n                            "evaluation_split": "legacy_validation_only", "device": device,\n                            "gpu": torch.cuda.get_device_name(0) if device != "cpu" else None,\n                            "normalization": "saved training-only percentile parameters", "input_policy": policy,\n                            "runtime_seconds": time.monotonic() - started,\n                            "versions": {p: importlib.metadata.version(p) for p in ("torch", "ultralytics", "pycocotools", "numpy")}}\n    save(f"{model_id}_size_ap.json", report)\n    event(f"D1-{model_id}", "measured_legacy_validation", evidence=f"{model_id}_size_ap.json",\n          observation=report["roadmap_strict"]["size_metrics"],\n          evidence_quality="original-pixel ground truth and predictions from hash-matched historical checkpoint",\n          hypothesis="size diagnostics measured; no training improvement tested",\n          decision="use measured size support to design P-series; require fixed-fold training for promotion",\n          smallest_next_test="matched P1 higher-resolution evaluation after baseline CV")\n\n\ndef main():\n    data = Path(kagglehub.dataset_download("itsasup/hyperspectral-d01-base-data-col-fix"))\n    base = one(data, "hyperspectral-2026")\n    backup = Path(kagglehub.dataset_download("itsasup/working-backup-plan-c-hod"))\n    plan_c = one(backup, "hsi_plan_c_ratio")\n    inventory = [{"path": str(p.relative_to(data)), "bytes": p.stat().st_size} for p in data.rglob("*") if p.is_file()]\n    save("data_inventory.json", inventory)\n    with zipfile.ZipFile(OUT / "training_annotation_metadata.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in (base / "data_train/data_train/Annotations/VIS").rglob("*.xml"):\n            archive.write(path, str(path.relative_to(base)))\n        archive.write(base / "class.txt", "class.txt")\n    command = [sys.executable, "/kaggle/working/build_folds.py", "--xml-dir", str(base / "data_train/data_train/Annotations/VIS"),\n               "--cube-dir", str(base / "data_train/data_train/VIS"), "--class-file", str(base / "class.txt"),\n               "--legacy-split", str(plan_c / "split_manifest.json"), "--output-dir", str(OUT / "folds"), "--seed", "42"]\n    result = subprocess.run(command, text=True, capture_output=True)\n    (OUT / "fold_build.log").write_text(result.stdout + result.stderr)\n    event("V1", "folds_built" if result.returncode == 0 else "failed_requires_diagnosis",\n          evidence="fold_build.log", returncode=result.returncode,\n          decision="review grouping metadata and fold diagnostics before training", smallest_next_test="group-evidence audit")\n    jobs = [\n        ("rtdetr_plan_c", "dataset", "itsasup/working-backup-plan-c-hod", "b4a54b0f5ad05a25fae96daf070e53286852f15c7091dd3fa13af2113afb9012", "rtdetr", "hsi_plan_c_ratio"),\n        ("rtdetr_plan_b", "mounted", None, "87292ca78adee9163a1dca6d8aafaa48ebb7a550e2347e2d265d2db1ad493395", "rtdetr", "hsi_plan_b_v2"),\n        ("yolo_plan_a", "mounted", None, "52800d6c16b33155e34d6e53ecdb31bb009ed0e1356d1777ca67b29d0509e291", "yolo", "hsi_plan_a_recovery"),\n    ]\n    for model_id, kind, source, expected_hash, family, run_dir in jobs:\n        try:\n            artifact = backup if kind == "dataset" else Path("/kaggle/input")\n            diagnostics(base, model_id, artifact, expected_hash, family, run_dir)\n        except Exception as error:\n            (OUT / f"{model_id}_failure.txt").write_text(traceback.format_exc())\n            event(f"D1-{model_id}", "failed_requires_diagnosis", error=str(error),\n                  decision="repair exact failure and rerun; do not drop required experiment",\n                  smallest_next_test="resolve recorded failure", evidence=f"{model_id}_failure.txt")\n    with zipfile.ZipFile("/kaggle/working/roadmap_day1_evidence.zip", "w", zipfile.ZIP_DEFLATED) as archive:\n        for path in OUT.rglob("*"):\n            if path.is_file():\n                archive.write(path, str(path.relative_to(OUT)))\n    print("Evidence package: /kaggle/working/roadmap_day1_evidence.zip", flush=True)\n\n\nif __name__ == "__main__":\n    main()\n')


In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, '/kaggle/working/run_day1_diagnostics.py'], check=True)
